<a href="https://colab.research.google.com/github/Melissa-Etes/Sprint2_grupo54/blob/main/EC_Sprint2_HospiDataSUS_Clusterizacao_Grupo54.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Etapa Clusters - DataSUS - Grupo 54 (Sprint 2 FIAP)

#### Imports

In [9]:

import pandas as pd
import numpy as np

##### Carregando Dados do GitHub

In [10]:
dados = pd.read_csv('https://raw.githubusercontent.com/Melissa-Etes/Sprint2_grupo54/main/data/internacoes_tratado.csv')
dados.head()

,COD_MUNICIPIO,MUNICIPIO,QTD_INTERNACOES,PERMANENCIA_MEDIA,TAXA_MORTALIDADE_PCT,VALOR_MEDIO_INTERNACAO,QTD_ESTAB_HOSPITALARES,POPULACAO,INTERNACOES_POR_MIL_HAB
0,353760,Peruíbe,263,5.55,11.03,1472.34,0,68352,3.85
1,350190,Amparo,336,6.09,10.71,1931.03,0,68008,4.94
2,351020,Capão Bonito,261,3.50,10.73,1310.42,0,46337,5.63
3,350660,Biritiba Mirim,136,5.42,11.03,1676.72,0,29683,4.58
4,353600,Parapuã,57,7.28,8.77,1149.48,0,10580,5.39


,COD_MUNICIPIO,MUNICIPIO,QTD_INTERNACOES,PERMANENCIA_MEDIA,TAXA_MORTALIDADE_PCT,VALOR_MEDIO_INTERNACAO,QTD_ESTAB_HOSPITALARES,POPULACAO,INTERNACOES_POR_MIL_HAB
0,353760,Peruíbe,263,5.55,11.03,1472.34,0,68352,3.85
1,350190,Amparo,336,6.09,10.71,1931.03,0,68008,4.94
2,351020,Capão Bonito,261,3.50,10.73,1310.42,0,46337,5.63
3,350660,Biritiba Mirim,136,5.42,11.03,1676.72,0,29683,4.58
4,353600,Parapuã,57,7.28,8.77,1149.48,0,10580,5.39


##### Conferir os dados

In [11]:
dados.shape
dados.isna().sum()

,0
COD_MUNICIPIO,0
MUNICIPIO,0
QTD_INTERNACOES,0
PERMANENCIA_MEDIA,0
TAXA_MORTALIDADE_PCT,0
VALOR_MEDIO_INTERNACAO,0
QTD_ESTAB_HOSPITALARES,0
POPULACAO,0
INTERNACOES_POR_MIL_HAB,0


,0
COD_MUNICIPIO,0
MUNICIPIO,0
QTD_INTERNACOES,0
PERMANENCIA_MEDIA,0
TAXA_MORTALIDADE_PCT,0
VALOR_MEDIO_INTERNACAO,0
QTD_ESTAB_HOSPITALARES,0
POPULACAO,0
INTERNACOES_POR_MIL_HAB,0


In [12]:
dados = dados.dropna(subset=['POPULACAO'])
dados['QTD_ESTAB_HOSPITALARES'] = dados['QTD_ESTAB_HOSPITALARES'].fillna(0)

### Selecionar as features

In [13]:
features = dados[['QTD_INTERNACOES', 'PERMANENCIA_MEDIA', 'TAXA_MORTALIDADE_PCT',
                   'QTD_ESTAB_HOSPITALARES', 'INTERNACOES_POR_MIL_HAB']]
features.head()

,QTD_INTERNACOES,PERMANENCIA_MEDIA,TAXA_MORTALIDADE_PCT,QTD_ESTAB_HOSPITALARES,INTERNACOES_POR_MIL_HAB
0,263,5.55,11.03,0,3.85
1,336,6.09,10.71,0,4.94
2,261,3.50,10.73,0,5.63
3,136,5.42,11.03,0,4.58
4,57,7.28,8.77,0,5.39


,QTD_INTERNACOES,PERMANENCIA_MEDIA,TAXA_MORTALIDADE_PCT,QTD_ESTAB_HOSPITALARES,INTERNACOES_POR_MIL_HAB
0,263,5.55,11.03,0,3.85
1,336,6.09,10.71,0,4.94
2,261,3.50,10.73,0,5.63
3,136,5.42,11.03,0,4.58
4,57,7.28,8.77,0,5.39


### Pipeline de padronização + PCA

In [14]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

SEED = 1224
np.random.seed(SEED)

pca_pipeline = Pipeline([('scaler', StandardScaler()), ('PCA', PCA(n_components=2, random_state=SEED))])
embedding_pca = pca_pipeline.fit_transform(features)
projection = pd.DataFrame(columns=['x', 'y'], data=embedding_pca)
projection.head()

,x,y
0,-0.699530,0.696140
1,-0.148790,0.782673
2,-0.665035,-0.053534
3,-0.484692,0.589860
4,0.489611,0.804719


,x,y
0,-0.699530,0.696140
1,-0.148790,0.782673
2,-0.665035,-0.053534
3,-0.484692,0.589860
4,0.489611,0.804719


### Definir o número ideal de clusters

In [15]:
import plotly.express as px
from sklearn.cluster import KMeans

inercia = []
for k in range(1, 10):
    km = KMeans(n_clusters=k, random_state=SEED, n_init=10)
    km.fit(embedding_pca)
    inercia.append(km.inertia_)

fig = px.line(x=list(range(1,10)), y=inercia, markers=True,
              title='Método do Cotovelo', labels={'x':'Número de clusters (k)', 'y':'Inércia'})
fig.show()

In [16]:
fig.write_image("metodo_cotovelo.png")

## Rodar o K-Means com o k escolhido

In [17]:
K_ESCOLHIDO = 3  # ajuste conforme o gráfico do cotovelo

kmeans = KMeans(n_clusters=K_ESCOLHIDO, random_state=SEED, n_init=10)
projection['cluster'] = kmeans.fit_predict(embedding_pca).astype(str)
projection['municipio'] = dados['MUNICIPIO'].values
projection.head()

,x,y,cluster,municipio
0,-0.699530,0.696140,0,Peruíbe
1,-0.148790,0.782673,0,Amparo
2,-0.665035,-0.053534,0,Capão Bonito
3,-0.484692,0.589860,0,Biritiba Mirim
4,0.489611,0.804719,0,Parapuã


### Plotar os clusters

In [19]:
fig = px.scatter(
    projection, x='x', y='y', color='cluster',
    hover_data=['municipio'],
    title='Clusters de Municípios por Perfil de Sobrecarga Hospitalar'
)
fig.update_traces(marker=dict(size=8, opacity=0.9, line=dict(width=0.5, color='#121212')))
fig.show()

In [20]:
fig.write_image("dispersao_clusters_municipios.png")

### Interpretar os clusters


In [21]:
dados['cluster'] = projection['cluster'].values
dados.groupby('cluster')[['QTD_INTERNACOES', 'PERMANENCIA_MEDIA', 'TAXA_MORTALIDADE_PCT', 'INTERNACOES_POR_MIL_HAB']].mean()

,QTD_INTERNACOES,PERMANENCIA_MEDIA,TAXA_MORTALIDADE_PCT,INTERNACOES_POR_MIL_HAB
cluster,,,,
0,535.956204,5.597263,10.683869,4.532482
1,223.000000,21.180000,1.305000,23.740000
2,135.926829,4.647127,4.710108,6.149377
